# 0. Problem
## 196. Delete Duplicate Emails — Easy
Keep only the smallest `id` for each email. LeetCode/MySQL uses a destructive `DELETE`; pandas/PySpark demonstrate the equivalent retained DataFrame.

Official: https://leetcode.com/problems/delete-duplicate-emails/

# 1. Setup

In [ ]:
import pandas as pd
person_rows=[(1,"john@example.com"),(2,"bob@example.com"),(3,"john@example.com"),(4,"bob@example.com"),(5,"alice@example.com")]
person_pd=pd.DataFrame(person_rows,columns=["id","email"])
person_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark=SparkSession.builder.getOrCreate()
person_spark=spark.createDataFrame(person_rows,["id","email"])
person_spark.createOrReplaceTempView("Person")

# 2. SQL Solution

Canonical MySQL mutation:

```sql
DELETE p1
FROM Person p1
JOIN Person p2
  ON p1.email=p2.email
 AND p1.id>p2.id;
```

The runnable notebook query below previews the retained rows safely.

In [ ]:
sql_result=spark.sql("""
SELECT id,email
FROM (
    SELECT id,email,ROW_NUMBER() OVER(PARTITION BY email ORDER BY id) AS rn
    FROM Person
) x
WHERE rn=1
ORDER BY id
""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
result_pd=(person_pd.sort_values("id").drop_duplicates(subset="email",keep="first").sort_values("id").reset_index(drop=True))
result_pd

# 4. PySpark Solution

In [ ]:
w=Window.partitionBy("email").orderBy("id")
result_spark=(person_spark.withColumn("rn",F.row_number().over(w)).filter(F.col("rn")==1).drop("rn").orderBy("id"))
result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| keep smallest/group | self-delete / `ROW_NUMBER()` | sort + `.drop_duplicates()` | Window `row_number()` |
| destructive mutation | `DELETE` | cleaned DataFrame | immutable DataFrame |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Person

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: person_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: person_spark